# How to build tokenizer for LLM
### Byte-Pair Encoding from zero to hero —  Python

This notebook follows the workshop slides end to end. Every code block that appears on a slide is here as a real, runnable cell — plus a live example after every concept, using a real Polish text (Żeromski's *Siłaczka*) as the running corpus.

**Plan (from slide 2):**
1. Why tokenize at all
2. Naive tokenizer and its limits
3. BPE — algorithm theory
4. BPE from scratch in Python (the heart of the workshop)
5. Byte-level BPE — why not characters
6. Case study - out of scope for now
7. Production traps - out of scope for now

**Goal:** understand *why* tokenization looks the way it does, and write BPE from scratch.


## What is a tokenizer?
A **tokenizer** is a tool that breaks text into smaller pieces, called tokens, so a language model can process it. Tokens can be whole words, parts of words, or individual characters, depending on the tokenizer.
For example, "unbelievable" might split into tokens like "un", "believ", and "able", while common short words like "the" or "cat" usually stay as single tokens.
Tokenizers matter because models don't read raw text; they work with sequences of numbers, where each token maps to an ID in a fixed vocabulary. The tokenizer converts text to token IDs before the model runs, and converts IDs back to text for the output. Tokenization also affects things like context window limits (which are measured in tokens, not characters or words) and how efficiently a model handles different languages or unusual words.

## 1. Why doesn't an LLM read text character by character? (slide 3)

Three arguments:

1. **Sequence length** — the context window is finite (e.g. 128k tokens). Using a character as the unit means 4–5x longer sequences. It is then more expensive and less efficient.
2. **Words don't work either** — if we take words, the vocabulary explodes, especially with inflection.
3. **Subword is the compromise** — units between a character and a whole word. Like you got word unbelievable, then subwords could be "un", "believ" and "able".

## Vocabulary explosion in inflection example(slide 4)

One concept, a dozen forms: *dom · domu · domowi · domem · domach · domowy · domowego · ...* (Polish for "house").

> **Key sentence from the slide:** a tokenizer is lossy compression at the representation level — it just learns which sequences of bytes occur together often enough that it's worth giving them one identifier.

Small illustration: if a word-level tokenizer needs one vocabulary slot per inflected form, one Polish noun alone can cost you 10+ slots. A subword tokenizer instead stores the stem once and reuses small suffix pieces across thousands of different words.

In [69]:
# Quick illustration of the vocabulary-explosion point from slide 4 (not on the slide itself, just for intuition)
inflected_forms = ["dom", "domu", "domowi", "domem", "domach", "domowy", "domowego"]

word_level_vocab = set(inflected_forms)
print(f"Word-level tokenizer needs {len(word_level_vocab)} separate vocabulary entries for ONE concept.")

# a subword view: stem + reusable suffixes
stem = "dom"
suffixes = {f[len(stem):] for f in inflected_forms}
print(f"Subword view: 1 stem ('{stem}') + {len(suffixes)} suffix pieces {sorted(suffixes)}")
print("Those same suffix pieces (u, owi, em, ach, owy, owego) get reused across MANY other Polish words too.")


Word-level tokenizer needs 7 separate vocabulary entries for ONE concept.
Subword view: 1 stem ('dom') + 7 suffix pieces ['', 'ach', 'em', 'owego', 'owi', 'owy', 'u']
Those same suffix pieces (u, owi, em, ach, owy, owego) get reused across MANY other Polish words too.


## 2. Naive tokenizer — simplest possible version (slide 5)

Just a regex: split into "word chunks" and "everything else" (punctuation, symbols).

In [70]:
import re

def naive_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

print(naive_tokenize("Nie lubię tokenizacji w językach fleksyjnych."))


['Nie', 'lubię', 'tokenizacji', 'w', 'językach', 'fleksyjnych', '.']


## Exercise: where does it break? (slide 6)

Run the naive tokenizer on four tricky cases and see the real breakage:

- `"don't stop believing"` — apostrophe
- `"Kupiłem 3,5 kg jabłek za 12.99 zł"` — numbers get mangled
- `"super!!!"` — emoji + repeated punctuation
- `"niesamowicie-fantastycznie-dobre"` — hyphens

**Bridge to BPE:** every one of these needs its own hand-written regex rule. That doesn't scale and doesn't generalize to a new language — we need an approach that *learns* its units from data.

In [71]:
test_cases = [
    "don't stop believing",
    "Kupiłem 3,5 kg jabłek za 12.99 zł",
    "super!!!",
    "niesamowicie-fantastycznie-dobre",
]

for t in test_cases:
    print(f"{t!r}\n  -> {naive_tokenize(t)}\n")


"don't stop believing"
  -> ['don', "'", 't', 'stop', 'believing']

'Kupiłem 3,5 kg jabłek za 12.99 zł'
  -> ['Kupiłem', '3', ',', '5', 'kg', 'jabłek', 'za', '12', '.', '99', 'zł']

'super!!!'
  -> ['super', '!', '!', '!']

'niesamowicie-fantastycznie-dobre'
  -> ['niesamowicie', '-', 'fantastycznie', '-', 'dobre']



## 3. Byte-Pair Encoding — the algorithm (slide 7)

1. Start from the smallest units (bytes or characters).
2. Count the frequency of every adjacent pair of tokens.
3. Find the most frequent pair, merge it into a new token. Record the rule (`merge`).
4. Repeat steps 2–3 or `N` times.

`N` = target vocabulary size − size of the base alphabet.

## 3.5 Byte-Pair Encoding - example (slide 8)

Let's assume we got a corpus "aaabdaaabac".

First pair  -> "aa" after merge -> "Z", corpus becomes "ZabdZabac"
Second pair -> "ab" after merge -> "Y", corpus becomes ZYdZYac"
Third pair  -> "ZY" after merge -> "X", corpus becomes XdXac

After 3 merges, the dictionary is {a, b, c, d, Z, Y, X}.

Explanation:
**corpus** - training text, written as a sequence of individual characters at the start. It gets rewritten every merge step.
**pair** - the most frequently occuring pair of adjacent symbols in the current corpus. BPE scans the whole corpus and counts every adjacent pair, then picks the most common one.
**merge** - the rule that says "replace this pair with one new symbol." That new symbol (Z, then Y, then X) gets added to vocabulary.

Walking through it:
1. Corpus "aaabdaaabac". The most common pair is "aa" (appears 4 times). *Merge rule:* "aa" -> "Z". Rewriting the corpus replacing every "aa" gives "ZabdZabac".
2. Count pair again in "ZabdZabac". The most common pair is "ab". *Merge rule:* "ab" -> "Y". Corpus becomes "ZYdZYac".
3. Count pair again in "ZYdZYac". The most common pair is "ZY". *Merge rule:* "ZY" -> "X". Corpus becomes "XdXac".

This is exactly how real tokenizers are trained, but just a tiny toy example. What is the purpose then? TO build a **vocabulary of reusable subword pieces** that can efficiently encode *any* text later, not just the training corpus.

## 4. BPE from scratch — the heart of the workshop (slides 9–13)

So the process looks like 4 separate functions that we can test independently:

- **4.1** counting pairs
- **4.2** merge
- **4.3** training loop
- **4.4** encoder for new text

We build the first two now so we can immediately run the tiny live example from slide 8, then scale up to the real corpus.

### 4.1 — counting pairs (slide 10)

BPE always merges the **most frequent pair**, because that gives the biggest reduction in sequence length per merge.
Without counting, you have no principled way to choose which pair to merge next, you would just be guessing.
Example: on "aaabdaaabac", counting gives "aa":4, "ab":2, "bd":1, "da":1, "ba":1, "ac":1. The count tells you that "aa" is
the winner, so it gets merged first, not because it is first in the string, but because it is the most common.

In [14]:
def get_pair_counts(token_ids):
    """Zlicz wystąpienia każdej sąsiadującej pary tokenów."""
    counts = {}
    for pair in zip(token_ids, token_ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

assert get_pair_counts([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
print("get_pair_counts OK")


get_pair_counts OK


### 4.2 — merge (slide 11)

Counting only tells you what to merge. You still need to actually rewrite the corpus so the new symbol exists and future "pair counts" runs can be based on it. This is also what lets merges stack, like "ZY" -> "X", which only makes sense once "Z" and "Y" already exist.

In [15]:
def merge(token_ids, pair, new_id):
    """Zamień wszystkie wystąpienia pary na nowy token."""
    new_ids = []
    i = 0
    while i < len(token_ids):
        if i < len(token_ids) - 1 and (token_ids[i], token_ids[i+1]) == pair:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(token_ids[i])
            i += 1
    return new_ids

assert merge([1, 2, 3, 1, 2], (1, 2), 99) == [99, 3, 99]
print("merge OK")


merge OK


### Live example — slide 8

Reproducing the exact toy example from the slide with the two functions we just wrote, over **characters** (not bytes yet — that distinction is section 5).

In [16]:
corpus = "aaabdaaabac"
toy_ids = list(corpus)  # characters as the starting "tokens"
toy_merges = []  # list of (pair, new_symbol), in order learned

for step in range(3):
    counts = get_pair_counts(toy_ids)
    best_pair = max(counts, key=counts.get)
    new_symbol = "ZYX"[step]  # Z, then Y, then X — matches the slide
    toy_ids = merge(toy_ids, best_pair, new_symbol)
    toy_merges.append((best_pair, new_symbol))
    print(f"{best_pair} -> {new_symbol}\t corpus: {''.join(toy_ids)}")

vocab = set("abcd") | {sym for _, sym in toy_merges}
print(f"\nThree merges, vocab: {sorted(vocab)}")
assert ''.join(toy_ids) == "XdXac"
print("Matches the slide's final corpus 'XdXac'.")


('a', 'a') -> Z	 corpus: ZabdZabac
('Z', 'a') -> Y	 corpus: YbdYbac
('Y', 'b') -> X	 corpus: XdXac

Three merges, vocab: ['X', 'Y', 'Z', 'a', 'b', 'c', 'd']
Matches the slide's final corpus 'XdXac'.


> **Insight (slide 8):** BPE training = learning a vocabulary and merge rules. Tokenization = applying those rules in the same order in which they were learned.

### 4.3 — training loop (slide 12)

Note the key detail: we start from `text.encode("utf-8")` — **raw bytes**, not characters. `next_id` starts at 256 because 0–255 are already taken by byte values.

Why training loop is needed? Merge only creates one new token. Real tokenizer needs thousands of merges to build useful vocabulary, so you need to repeat count -> pick most common pair -> merge, and do that over and over, while recording each rule **in the order it was created** as the order matters for the encoder!

In [ ]:
def train_bpe(text, vocab_size):
    token_ids = list(text.encode("utf-8"))  # start: surowe bajty
    merges = {}  # (a, b) -> new_id
    next_id = 256  # 0-255 zajęte przez bajty

    for i in range(vocab_size - 256):
        counts = get_pair_counts(token_ids)
        if not counts:
            break
        best_pair = max(counts, key=counts.get)
        token_ids = merge(token_ids, best_pair, next_id)
        merges[best_pair] = next_id
        next_id += 1
    return token_ids, merges

print("train_bpe defined")

train_bpe defined


### Now let's run it on a real corpus: *Siłaczka* by Stefan Żeromski

In [74]:
CORPUS_PATH = "krzyzacy_tom_1.txt"

with open(CORPUS_PATH, encoding="utf-8") as f:
    my_text = f.read()

#### Step 1 — convert text to raw bytes

We encode as UTF-8: every character becomes 1–4 bytes. Polish characters like ą, ę, ó each take 2 bytes. The result is a flat list of integers 0–255 — our initial 256-token vocabulary.

In [75]:
text_bytes = list(my_text.encode("utf-8"))

#### Sanity check — peek at the first 100 bytes

In [76]:
print(text_bytes[:100])

[72, 101, 110, 114, 121, 107, 32, 83, 105, 101, 110, 107, 105, 101, 119, 105, 99, 122, 10, 10, 75, 114, 122, 121, 197, 188, 97, 99, 121, 10, 84, 111, 109, 32, 73, 10, 10, 73, 83, 66, 78, 32, 57, 55, 56, 45, 56, 51, 45, 50, 56, 56, 45, 50, 56, 49, 51, 45, 49, 10, 10, 10, 10, 10, 82, 111, 122, 100, 122, 105, 97, 197, 130, 32, 112, 105, 101, 114, 119, 115, 122, 121, 10, 10, 87, 32, 84, 121, 197, 132, 99, 117, 44, 32, 119, 32, 103, 111, 115, 112]


#### Inspect the top 20 most frequent pairs before merging anything

This confirms the algorithm works and previews what the first merges will look like (often spaces + common letter combinations).

In [77]:
pair_counts = get_pair_counts(text_bytes)
print(pair_counts)

{(72, 101): 67, (101, 110): 1554, (110, 114): 8, (114, 121): 1661, (121, 107): 399, (107, 32): 1588, (32, 83): 478, (83, 105): 28, (105, 101): 16610, (110, 107): 369, (107, 105): 2495, (101, 119): 1013, (119, 105): 5763, (105, 99): 1606, (99, 122): 5208, (122, 10): 1, (10, 10): 3836, (10, 75): 36, (75, 114): 344, (114, 122): 7213, (122, 121): 5632, (121, 197): 2489, (197, 188): 6205, (188, 97): 547, (97, 99): 1778, (99, 121): 670, (121, 10): 28, (10, 84): 142, (84, 111): 230, (111, 109): 1116, (109, 32): 4133, (32, 73): 177, (73, 10): 1, (10, 73): 133, (73, 83): 2, (83, 66): 2, (66, 78): 3, (78, 32): 1, (32, 57): 1, (57, 55): 3, (55, 56): 2, (56, 45): 4, (45, 56): 2, (56, 51): 2, (51, 45): 4, (45, 50): 4, (50, 56): 4, (56, 56): 2, (56, 49): 2, (49, 51): 5, (45, 49): 2, (49, 10): 1, (10, 82): 43, (82, 111): 97, (111, 122): 1054, (122, 100): 447, (100, 122): 4016, (122, 105): 3674, (105, 97): 3224, (97, 197): 6075, (197, 130): 15453, (130, 32): 4631, (32, 112): 9224, (112, 105): 1213, (1

In [78]:
# Get pair frequencies
pair_counts = get_pair_counts(text_bytes)

# Sort by frequency, descending
sorted_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)

# Show the most frequent pair and the top 20 pairs
print("Most frequent pair:")
top_pair, top_pair_count = sorted_pairs[0]
print(f"Pair: {top_pair}, Count: {top_pair_count}")

print("\nTop 40 frequent pairs:")
for (pair, count) in sorted_pairs[:40]:
    try:
        chars = bytes([pair[0], pair[1]]).decode('utf-8', errors='replace')
    except Exception:
        chars = "<?>"
    print(f"Pair: {pair}, Count: {count}, As chars: {chars!r}")

Most frequent pair:
Pair: (105, 101), Count: 16610

Top 40 frequent pairs:
Pair: (105, 101), Count: 16610, As chars: 'ie'
Pair: (197, 130), Count: 15453, As chars: 'ł'
Pair: (44, 32), Count: 11741, As chars: ', '
Pair: (101, 32), Count: 11555, As chars: 'e '
Pair: (111, 32), Count: 9959, As chars: 'o '
Pair: (110, 105), Count: 9633, As chars: 'ni'
Pair: (97, 32), Count: 9529, As chars: 'a '
Pair: (32, 112), Count: 9224, As chars: ' p'
Pair: (105, 32), Count: 8654, As chars: 'i '
Pair: (196, 153), Count: 8644, As chars: 'ę'
Pair: (32, 110), Count: 8031, As chars: ' n'
Pair: (32, 119), Count: 7694, As chars: ' w'
Pair: (32, 115), Count: 7431, As chars: ' s'
Pair: (32, 122), Count: 7227, As chars: ' z'
Pair: (114, 122), Count: 7213, As chars: 'rz'
Pair: (105, 196), Count: 6338, As chars: 'i�'
Pair: (197, 188), Count: 6205, As chars: 'ż'
Pair: (97, 197), Count: 6075, As chars: 'a�'
Pair: (99, 104), Count: 6003, As chars: 'ch'
Pair: (121, 32), Count: 5976, As chars: 'y '
Pair: (196, 133), C

#### Do a single merge by hand

Apply exactly one merge — the most frequent pair — so you can watch the corpus length shrink. This is just the manual version of one loop iteration; the full training loop follows below.

In [80]:
new_token_id = 256
text_merged = merge(text_bytes, top_pair, new_token_id)

print(f"Merged pair: {top_pair} -> {repr(bytes(top_pair).decode('utf-8', errors='replace'))} => token {new_token_id}")
print(f"Length before: {len(text_bytes)}")
print(f"Length after:  {len(text_merged)}")
print(f"Reduction:     {len(text_bytes) - len(text_merged)} tokens")

Merged pair: (105, 101) -> 'ie' => token 256
Length before: 732128
Length after:  715518
Reduction:     16610 tokens


#### Run the full training loop

With `get_pair_counts` and `merge` in place, `train_bpe` runs the whole loop for us — *count every pair → merge the most frequent → record the rule*, repeated `NUM_OF_MERGES` times. It returns two things:

- **`silaczka_merges`** — the learned rules `{pair: new_id}`, in learning order. **This is the trained tokenizer**, reused later by `encode` and `decode`.
- **`silaczka_tokens`** — the corpus after all merges; a by-product kept only to measure how much the sequence shrank.

In [82]:
NUM_OF_MERGES = 1000
silaczka_tokens, silaczka_merges = train_bpe(my_text, vocab_size=256 + NUM_OF_MERGES)

reduction = len(text_bytes) - len(silaczka_tokens)
print(f"Trained {len(silaczka_merges)} merges (vocab size {256 + NUM_OF_MERGES}).")
print(f"Corpus: {len(text_bytes)} bytes -> {len(silaczka_tokens)} tokens "
      f"({100 * reduction / len(text_bytes):.1f}% reduction).")

Trained 1000 merges (vocab size 1256).
Corpus: 732128 bytes -> 245803 tokens (66.4% reduction).


#### Inspect the learned BPE merges

`silaczka_merges` only stores pairs of numeric IDs, and those IDs can themselves be earlier merges (e.g. `(110, 257)` = `n` + the `ie` token). To read them, we rebuild each token's bytes in learning order and decode to text — revealing the actual Polish subword pieces BPE discovered (`ie`, `nie`, `cz`, `rz`, `ch`, ...).

In [38]:
# Rebuild each learned token's bytes in learning order, then decode to text so we can read it.
# (Same idea as build_vocab in the decoder section below — inlined here just to inspect the merges.)
token_bytes = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in silaczka_merges.items():
    token_bytes[new_id] = token_bytes[a] + token_bytes[b]
    piece = token_bytes[new_id].decode("utf-8", errors="replace")
    print(f"token {new_id}: ({a}, {b}) -> {piece!r}")

token 256: (197, 130) -> 'ł'
token 257: (105, 101) -> 'ie'
token 258: (44, 32) -> ', '
token 259: (97, 32) -> 'a '
token 260: (111, 32) -> 'o '
token 261: (196, 153) -> 'ę'
token 262: (196, 133) -> 'ą'
token 263: (105, 32) -> 'i '
token 264: (110, 257) -> 'nie'
token 265: (99, 122) -> 'cz'
token 266: (99, 104) -> 'ch'
token 267: (114, 122) -> 'rz'
token 268: (32, 115) -> ' s'
token 269: (97, 256) -> 'ał'
token 270: (32, 119) -> ' w'
token 271: (32, 112) -> ' p'
token 272: (226, 128) -> '�'
token 273: (32, 122) -> ' z'
token 274: (197, 155) -> 'ś'
token 275: (197, 188) -> 'ż'
token 276: (105, 261) -> 'ię'
token 277: (195, 179) -> 'ó'
token 278: (115, 122) -> 'sz'
token 279: (110, 105) -> 'ni'
token 280: (32, 100) -> ' d'
token 281: (115, 116) -> 'st'
token 282: (111, 119) -> 'ow'
token 283: (110, 97) -> 'na'
token 284: (100, 122) -> 'dz'
token 285: (114, 97) -> 'ra'
token 286: (46, 32) -> '. '
token 287: (196, 135) -> 'ć'
token 288: (10, 10) -> '\n\n'
token 289: (121, 32) -> 'y '
token 

In [83]:
import json

# Export the trained tokenizer (the merge rules) to JSON.
# Tuple keys aren't valid JSON, so each rule (a, b) -> new_id is flattened to [a, b, new_id].
# Dict insertion order == learning order, and a JSON list preserves it, so the round-trip stays faithful.
MERGES_PATH = "silaczka_tokenizer_merges.json"

with open(MERGES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {"merges": [[a, b, new_id] for (a, b), new_id in silaczka_merges.items()]},
        f,
    )

print(f"Saved {len(silaczka_merges)} merges to {MERGES_PATH}")

# Sanity check: load it back and confirm the reconstructed dict is identical.
with open(MERGES_PATH, encoding="utf-8") as f:
    loaded_merges = {(a, b): new_id for a, b, new_id in json.load(f)["merges"]}

assert loaded_merges == silaczka_merges, "round-trip mismatch!"
print("Round-trip OK — reloaded merges match silaczka_merges")


Saved 1000 merges to silaczka_tokenizer_merges.json
Round-trip OK — reloaded merges match silaczka_merges


## Encoder & Decoder — the two ways to *use* a trained tokenizer

Training is done: `silaczka_merges` holds the learned rules. But a model never reads or writes text — it only ever works with **sequences of token IDs**. So we need two functions that sit on the boundary between "human text" and "model IDs":

- **Encoder** — text → token IDs (used *before* the model runs)
- **Decoder** — token IDs → text (used *after* the model produces output)

They are exact inverses of each other, and both are driven entirely by `silaczka_merges`.

### Encoder (4.4 below)

**Why it's needed:** the whole point of *training* is to reuse the learned rules on text the tokenizer has **never seen**. The encoder takes new text, starts from its raw UTF-8 bytes, and repeatedly applies the merge rules until nothing more can be merged — turning the byte sequence into a much shorter sequence of token IDs.

**The subtle trap:** merges must be applied in the **exact order they were learned**, because later merges are built on top of earlier ones. At each step the encoder picks the pair whose merge was learned *earliest* (`min(..., key=lambda p: merges.get(p, inf))`) — not just the most frequent pair in this particular text.

Example (new, unseen text): take `"abaabac"`.
- Apply rule 1 (`aa→Z`): `"abaabac"` → `"abZbac"`
- Apply rule 2 (`ab→Y`): `"abZbac"` → `"YZbac"`
- Apply rule 3 (`ZY→X`): looks for `Z` followed by `Y`. Here we have `Y` followed by `Z` — reversed order — so it does **not** match. Final tokens: `{Y, Z, b, a, c}`.

Applying merges out of order would silently produce wrong tokens — so order is not optional.

### Decoder (bonus, further below)

**Why it's needed:** the model outputs token IDs, but we want readable text back. The decoder reverses the encoder. It relies on `build_vocab`, which walks the merge rules and reconstructs, for **every** token ID, the actual bytes it represents (a base byte for IDs 0–255, or the concatenation of two earlier tokens for learned IDs). `decode` then joins those bytes back together and decodes them as UTF-8.

A correct tokenizer round-trips: `decode(encode(text)) == text`. That equality is what proves the encoder and decoder agree on the same set of rules.


In [84]:
def encode(text, merges):
    token_ids = list(text.encode("utf-8"))
    while len(token_ids) >= 2:
        counts = get_pair_counts(token_ids)
        # para o najniższym ID merge'a = wyuczona najwcześniej
        pair = min(counts, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        token_ids = merge(token_ids, pair, merges[pair])
    return token_ids

print("encode defined")


encode defined


## Decoder (slide 14, bonus)

The decoder is split into two parts: first **build the vocabulary**, then **use it to decode**.

### Part 1 — `build_vocab`: building the vocabulary

In [85]:
def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), new_id in merges.items():
        vocab[new_id] = vocab[a] + vocab[b]
    return vocab

**What it does:** builds a lookup table mapping every token ID → the actual bytes it represents.
- The first line seeds IDs `0–255` with their single raw byte (every byte value is always a valid token).
- The loop walks the learned merges *in order*. Each merged token's bytes are just its two halves glued together — and because earlier merges are filled in first, `vocab[a]` and `vocab[b]` always already exist when needed.

**Why it's needed:** the token IDs are just *numbers* — `257` means nothing on its own. To turn IDs back into text, you need a table that says "ID 257 = these bytes." `build_vocab` constructs that table once, so decoding becomes a simple lookup instead of re-deriving each token from the merge rules every time. It's the bridge between the abstract IDs and real bytes.


### Part 2 — `decode`: turning IDs back into text

In [86]:
def decode(token_ids, vocab):
    return b"".join(vocab[i] for i in token_ids).decode("utf-8", errors="replace")

**What it does:** for each token ID, look up its bytes in `vocab`, concatenate them all into one byte string, then decode that whole byte string as UTF-8 **once at the very end**.

**Why it's needed:** `encode` only goes one way (text → IDs). A tokenizer is useless without the reverse — you need `decode` to read a model's output, since models emit token IDs, not text.

**Why decode at the end (not per token):** a single character like Polish `ł` or `ż` is multiple UTF-8 bytes, and those bytes can land in *different* tokens. Decoding each token's bytes individually would hit half-characters and produce garbage. Joining all bytes first, then decoding once, reassembles multi-byte characters correctly. `errors="replace"` is the safety net for any leftover incomplete byte.

A correct tokenizer round-trips: `decode(encode(text)) == text` — that equality is what proves the encoder and decoder agree on the same set of rules.

In [87]:
new_sentence = "Doktor jechał szybko przez ciemny las w stronę odległego miasta."
bpe_tokens = encode(new_sentence, silaczka_merges)

silaczka_vocab = build_vocab(silaczka_merges)
roundtrip = decode(bpe_tokens, silaczka_vocab)
print(f"Decoded back: {roundtrip!r}")
assert roundtrip == new_sentence
print("Round-trip OK: encode -> decode reproduces the original sentence exactly.")


Decoded back: 'Doktor jechał szybko przez ciemny las w stronę odległego miasta.'
Round-trip OK: encode -> decode reproduces the original sentence exactly.


## Use different text on trained data

Training only had to happen once. Its entire product — the learned merge rules — is already saved to `silaczka_tokenizer_merges.json`. Here we simulate **shipping** the tokenizer: load those rules from disk and apply them to a sentence that was never in the training corpus, with **no retraining**.

**What we need (nothing from training itself):**

- **`silaczka_tokenizer_merges.json`** — the saved rules; the trained tokenizer's *data*.
- **`get_pair_counts`, `merge`, `encode`** — the *code* that turns text → token IDs.
- **`build_vocab`, `decode`** — the *code* that turns token IDs → text.

We do **not** need `train_bpe`, the *Siłaczka* corpus, or any of the `silaczka_*` training variables — those did their job during training and are irrelevant now.

### Step 1 — Load the trained tokenizer from disk

The rules were exported earlier to `silaczka_tokenizer_merges.json`. Each rule `(a, b) -> new_id` was flattened to `[a, b, new_id]` because tuple keys aren't valid JSON. We rebuild the `{(a, b): new_id}` dict and preserve its order — list order = insertion order = learning order, which the encoder depends on.

In [88]:
import json

MERGES_PATH = "silaczka_tokenizer_merges.json"
with open(MERGES_PATH, encoding="utf-8") as f:
    loaded = json.load(f)

# JSON stored each rule flattened as [a, b, new_id]; rebuild the {(a, b): new_id} dict.
# List order == insertion order == learning order, which the encoder relies on.
trained_merges = {(a, b): new_id for a, b, new_id in loaded["merges"]}

print(f"Loaded {len(trained_merges)} merge rules from {MERGES_PATH}")
print("First 5 rules (pair -> new_id):", list(trained_merges.items())[:5])

Loaded 1000 merge rules from silaczka_tokenizer_merges.json
First 5 rules (pair -> new_id): [((105, 101), 256), ((197, 130), 257), ((44, 32), 258), ((111, 32), 259), ((97, 32), 260)]


### Step 2 — Encode new, unseen text

`encode` starts from the raw UTF-8 bytes of the new sentence and repeatedly applies the loaded rules — always the earliest-learned matching pair first — until nothing more can be merged. The output is a shorter list of token IDs. We compare byte count vs. token count to see the compression on text the tokenizer was **never trained on**.

In [89]:
CORPUS_PATH_2 = "mendel_gdanski.txt"

with open(CORPUS_PATH_2, encoding="utf-8") as f:
    new_text = f.read()

raw_bytes = list(new_text.encode("utf-8"))
new_tokens = encode(new_text, trained_merges)

print(f"Text:        {new_text!r}")
print(f"UTF-8 bytes: {len(raw_bytes)}")
print(f"BPE tokens:  {len(new_tokens)}")
print(f"Token IDs:   {new_tokens}")

reduction = len(raw_bytes) - len(new_tokens)
print(f"Compression: {reduction} fewer ({100 * reduction / len(raw_bytes):.1f}% shorter than raw bytes)")

Text:        'Od wczoraj jakiś niepokój panuje w uliczce. Stary Mendel dziwi się i częściej niż zwykle nakłada krótką fajkę, patrząc w okno. Tych ludzi nie widział on tu jeszcze. Gdzie idą? Po co przystają z robotnikami, śpieszącymi do kopania fundamentów pod nowy dom niciarza Greulicha? Skąd się tu wzięły te obszarpane wyrostki? Dlaczego patrzą tak po sieniach? Skąd mają pieniądze, że idą w pięciu do szynku?\n\nStary Mendel kręci głową, smokcząc mały, silnie wygięty wiśniowy cybuszek. On zna tak dobrze tę uliczkę cichą. Jej fizjonomję, jej ruch, jej głosy, jej tętno.\n\nWie, kiedy, zza którego węgła wyjrzy w dzień pogodny słońce; ile dzieci przebiegnie rankiem, drepcąc do ochronki, do szkoły; ile zwiędłych dziewcząt w ciemnych chustkach, z małymi blaszeczkami w ręku, przejdzie, po trzy, po cztery, do fabryki cygar na robotę; ile kobiet przystanie z koszami na starym, wytartym chodniku, pokazując sobie zakupione jarzyny, skarżąc się na drogość jaj, mięsa i masła; ilu wyrobników przecła

### Step 3 — Decode the IDs back to text

`build_vocab` turns the loaded rules into an `id -> bytes` lookup table; `decode` looks up every token's bytes, joins them, and decodes the whole thing as UTF-8. The `assert` proves the loaded-from-disk tokenizer round-trips brand-new text losslessly.

In [90]:
vocab = build_vocab(trained_merges)
decoded = decode(new_tokens, vocab)

print(f"Decoded back: {decoded!r}")
assert decoded == new_text
print("Round-trip OK: the loaded tokenizer encodes and decodes new text losslessly.")

Decoded back: 'Od wczoraj jakiś niepokój panuje w uliczce. Stary Mendel dziwi się i częściej niż zwykle nakłada krótką fajkę, patrząc w okno. Tych ludzi nie widział on tu jeszcze. Gdzie idą? Po co przystają z robotnikami, śpieszącymi do kopania fundamentów pod nowy dom niciarza Greulicha? Skąd się tu wzięły te obszarpane wyrostki? Dlaczego patrzą tak po sieniach? Skąd mają pieniądze, że idą w pięciu do szynku?\n\nStary Mendel kręci głową, smokcząc mały, silnie wygięty wiśniowy cybuszek. On zna tak dobrze tę uliczkę cichą. Jej fizjonomję, jej ruch, jej głosy, jej tętno.\n\nWie, kiedy, zza którego węgła wyjrzy w dzień pogodny słońce; ile dzieci przebiegnie rankiem, drepcąc do ochronki, do szkoły; ile zwiędłych dziewcząt w ciemnych chustkach, z małymi blaszeczkami w ręku, przejdzie, po trzy, po cztery, do fabryki cygar na robotę; ile kobiet przystanie z koszami na starym, wytartym chodniku, pokazując sobie zakupione jarzyny, skarżąc się na drogość jaj, mięsa i masła; ilu wyrobników przecł